In [19]:
from enum import IntEnum
from aiomoex import get_board_candles
import asyncio
import aiohttp
from datetime import datetime, timedelta
import pandas as pd


class IntervalEnum(IntEnum):
    MINUTE = 1
    TEN_MINUTES = 10
    HOUR = 60
    DAY = 24
    WEEK = 7
    MONTH = 31

# объявим аннотацию для удобства
StockData = list[dict[str, str | int | float]]

async def fetch_ticker_data(session: aiohttp.ClientSession, ticker: str, interval: IntervalEnum, start_date: str, end_date: str) -> dict[str, StockData]:
    '''Функция получает данные о торгах по заданному тикеру с *start_date* по *end_date* с интервалом *interval*, возвращая словарь, где ключом является тикер, а значением - данные

    Args:
        session (aiohttp.ClientSession): aiottp сессия для отпаравки запросов
        ticker (str): Имя тикера
        interval (IntervalEnum): Одно из доступных значений для интервала времени
        start_date (str): Начальная дата в формате yyyy-mm-dd
        end_date (str): Конечная дата в формате yyyy-mm-dd

    Returns:
        dict[str, list[dict[str, str | int | float]]]: Словарь, где ключ - тикер, а значение - данные, например {'SBER': sber_data}
    '''
    try:
        # получаем данные по переданному тикеру за указанный период
        res = await get_board_candles(session, ticker, interval, start_date, end_date)
        return {ticker: res}
    except Exception as e:
        print(f'Ошибка парсинга. Не удалось получить данные для {ticker}, {e}')
        return {ticker: []}

async def get_moex_data(tickers: list[str], start_date: datetime, end_date: datetime = datetime.now(), interval: IntervalEnum = IntervalEnum.DAY) -> dict[str, StockData]:

    if interval not in IntervalEnum:
        raise ValueError(f"Неверный интервал. Допустимые значения: {IntervalEnum}")
    
    end_date_formatted = end_date.strftime('%Y-%m-%d')
    start_date_formatted = start_date.strftime('%Y-%m-%d')

    async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(connect=5)) as session:
        # собираем корутины в список
        coros = [fetch_ticker_data(session, ticker, interval, start_date_formatted, end_date_formatted) for ticker in tickers]

        # 'собираем' результаты корутин - непосредственно парсинг
        stock_data = await asyncio.gather(*coros)

    # разворачиваем список словарей в один словарь, например: [ {'SBER': sber_data}, {'GAZP': gazp_data} ] -> { 'SBER': sber_data, 'GAZP': gazp_data }
    stock_data = {ticker: data for element in stock_data for ticker, data in element.items()}
    return stock_data


gold_ticker = 'GLDRUB_TOM'

tickers = [gold_ticker]

delta = timedelta(days=180)
start_date = datetime.now() - delta

stock_data = await get_moex_data(tickers, start_date=start_date, interval=IntervalEnum.MINUTE)

gold_df = pd.DataFrame(stock_data[gold_ticker])
gold_df['ticker'] = gold_ticker
gold_df

,ticker


In [18]:
gazp_df = pd.DataFrame(stock_data['GLDRUB_TOM'])
gazp_df

""
